# Food Delivery Demand Forecasting
### Predicting Daily Food Delivery Order Volumes

**Project 32 of 50 — Time Series Forecasting Portfolio**

## Project Overview

We build a daily food delivery demand time series from order records and forecast
the next 28 days using MLForecast.

| Attribute | Detail |
|---|---|
| **Kaggle slug** | `shrutimechlearn/zomato-orders-analysis` |
| **Primary file** | `zomato.csv` |
| **Target** | Daily order count |
| **Frequency** | Daily (D) |
| **Season length** | 7 (weekly) |
| **TS library** | MLForecast (LightGBM) |

## Learning Objectives

1. Aggregate raw order records into a daily demand series
2. Identify food-delivery-specific demand drivers: rain days, weekends, pay days
3. Apply MLForecast LightGBM for recursive 28-day forecasting
4. Diagnose day-of-week and payday-week patterns
5. Compare MLForecast against FLAML and statistical baselines
6. Discuss how seasonal patterns differ from restaurant sit-down vs delivery

## Problem Statement

Forecast daily food delivery orders for the next 28 days to inform driver capacity planning and kitchen partner load balancing.

## Why This Project Matters

Food delivery platforms must dispatch enough drivers 2–4 hours ahead of each meal period. Accurate volume forecasts translate directly into fewer missed deliveries, lower customer churn, and optimised driver pay.

## Dataset Overview

| Column | Description |
|---|---|
| `order_date` | Date of the order |
| `order_time` | Time of the order |
| `restaurant_id` | Originating restaurant |
| `total_cost` | Order value |

We aggregate by `order_date` to produce a daily order count series.

## Dataset Source & License

- **Kaggle**: https://www.kaggle.com/datasets/shrutimechlearn/zomato-orders-analysis
- **License**: CC0 (Public Domain) — educational use

## Environment Setup

In [ ]:
import subprocess, sys
for p in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
          "statsmodels","scikit-learn","lazypredict","flaml",
          "statsforecast","mlforecast","lightgbm"]:
    try:
        __import__(p.split("[")[0].replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats as scipy_stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
from mlforecast import MLForecast
import lightgbm as lgb
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14,5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration

In [ ]:
PROJECT     = "Food Delivery Demand Forecasting"
KAGGLE_SLUG = "shrutimechlearn/zomato-orders-analysis"
FREQ        = "D"
SEASON_LEN  = 7
HORIZON     = 28
TEST_SIZE   = HORIZON
VAL_SIZE    = HORIZON
RANDOM_SEED = 42
FLAML_SECS  = 120
DATA_DIR    = pathlib.Path("data/zomato")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
for k in ["KAGGLE_USERNAME","KAGGLE_KEY","KAGGLE_API_TOKEN"]:
    if os.environ.get(k):
        kaggle_ok = True
        print(f"Credential found: {k}")
        break
kj = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kj.exists():
    kaggle_ok = True
    print(f"kaggle.json found at {kj}")
if not kaggle_ok:
    print("WARNING: No Kaggle credentials found!")
    print("https://www.kaggle.com/settings -> Create New Token")
else:
    print("Credentials verified.")

## Dataset Download

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download("shrutimechlearn/zomato-orders-analysis"))
    print(f"Dataset at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system("kaggle datasets download -d shrutimechlearn/zomato-orders-analysis -p data/zomato --unzip")
    data_path = DATA_DIR

csvs = sorted(data_path.rglob("*.csv"), key=lambda f: f.stat().st_size, reverse=True)
print(f"CSV files ({len(csvs)}): {[f.name for f in csvs]}") 

## Load & Build Daily Order Count

In [ ]:
df_raw = pd.read_csv(csvs[0])
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(3))

In [ ]:
# Find date column
date_col = next((c for c in df_raw.columns if "date" in c.lower() or "order_dt" in c.lower()), df_raw.columns[0])
print(f"Date column: '{date_col}'")

df_raw[date_col] = pd.to_datetime(df_raw[date_col], infer_datetime_format=True, errors="coerce")
df_valid = df_raw.dropna(subset=[date_col]).copy()
df_valid["date"] = df_valid[date_col].dt.normalize()

ts_raw = df_valid.groupby("date").size().reset_index(name="order_count")
ts_raw = ts_raw.sort_values("date").reset_index(drop=True)
print(f"Daily series: {len(ts_raw)} days")
print(f"Range: {ts_raw['date'].min().date()} to {ts_raw['date'].max().date()}")
print(ts_raw["order_count"].describe().round(1))

## Data Validation

In [ ]:
print("DATA VALIDATION")
print("="*50)
full_range = pd.date_range(ts_raw["date"].min(), ts_raw["date"].max(), freq="D")
missing_dates = full_range.difference(ts_raw["date"])
print(f"Missing dates: {len(missing_dates)}")
zero_days = (ts_raw["order_count"] == 0).sum()
print(f"Zero-order days: {zero_days}")
mu, sig = ts_raw["order_count"].mean(), ts_raw["order_count"].std()
outliers = ts_raw[abs(ts_raw["order_count"]-mu) > 3*sig]
print(f"3-sigma outliers: {len(outliers)}")

## Exploratory Data Analysis

In [ ]:
ts_raw = ts_raw.set_index("date").reindex(full_range).rename_axis("date")
ts_raw["order_count"] = ts_raw["order_count"].interpolate("linear").round().astype("Int64")
ts_raw = ts_raw.reset_index()
ts_df = ts_raw.rename(columns={"date":"ds","order_count":"y"}).copy()
ts_df["y"] = ts_df["y"].astype(float)
print(f"Series: {len(ts_df)} days")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"],name="Daily orders",line=dict(color="#1565C0",width=1)))
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"].rolling(14,center=True).mean(),
    name="14-day MA",line=dict(color="#F44336",width=2.5,dash="dot")))
fig.update_layout(title="Daily Food Delivery Orders",xaxis_title="Date",yaxis_title="Orders/Day",
    template="plotly_white",legend=dict(orientation="h",y=-0.2))
fig.show()

In [ ]:
ts_df["dow"]   = ts_df["ds"].dt.dayofweek
ts_df["month"] = ts_df["ds"].dt.month
fig, axes = plt.subplots(1,2,figsize=(14,4))
ts_df.groupby("dow")["y"].mean().plot(kind="bar",ax=axes[0],color="#1565C0",edgecolor="black",alpha=0.85)
axes[0].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"],rotation=0)
axes[0].set_title("Mean Orders by Day of Week")
ts_df.groupby("month")["y"].mean().plot(kind="bar",ax=axes[1],color="#2E7D32",edgecolor="black",alpha=0.85)
axes[1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],rotation=30)
axes[1].set_title("Mean Orders by Month")
plt.tight_layout(); plt.show()
ts_df = ts_df.drop(columns=["dow","month"])

In [ ]:
ts_idx = ts_df.set_index("ds")["y"].asfreq("D")
decomp = seasonal_decompose(ts_idx, model="additive", period=7)
fig, axes = plt.subplots(4,1,figsize=(14,10),sharex=True)
for ax, s, t, c in zip(axes,
    [decomp.observed,decomp.trend,decomp.seasonal,decomp.resid.dropna()],
    ["Observed","Trend","Seasonal (weekly)","Residual"],
    ["#1565C0","#C62828","#2E7D32","#757575"]):
    s.plot(ax=ax,title=t,color=c)
plt.tight_layout(); plt.show()

fig2, axes2 = plt.subplots(1,2,figsize=(14,4))
plot_acf(ts_df["y"].dropna(),lags=49,ax=axes2[0],title="ACF")
plot_pacf(ts_df["y"].dropna(),lags=49,ax=axes2[1],title="PACF")
plt.tight_layout(); plt.show()

adf = adfuller(ts_df["y"].dropna())
print(f"ADF p={adf[1]:.4f} -> {'stationary' if adf[1]<0.05 else 'non-stationary'}") 

## Target Analysis

In [ ]:
print(ts_df["y"].describe().round(2))
fig, axes = plt.subplots(1,2,figsize=(12,4))
ts_df["y"].hist(bins=40,ax=axes[0],edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].set_title("Distribution of Daily Orders")
scipy_stats.probplot(ts_df["y"],dist="norm",plot=axes[1])
axes[1].set_title("Q-Q Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

In [ ]:
n = len(ts_df)
ts_train     = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_val       = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_test      = ts_df.iloc[n-TEST_SIZE:].copy()
ts_train_val = pd.concat([ts_train,ts_val]).reset_index(drop=True)
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max()   < ts_test["ds"].min()
print(f"Train : {len(ts_train):3d} days  |  Val : {len(ts_val):3d}  |  Test : {len(ts_test):3d}")

## Preprocessing

In [ ]:
lo = ts_train["y"].quantile(0.01); hi = ts_train["y"].quantile(0.99)
ts_train_w = ts_train.copy(); ts_train_w["y"] = ts_train_w["y"].clip(lo,hi)
ts_tv_w = pd.concat([ts_train_w,ts_val]).reset_index(drop=True)
print(f"Winsorisation [{lo:.0f}, {hi:.0f}]")

## Feature Engineering

In [ ]:
def make_features(df):
    df = df.copy()
    for lag in [1,2,3,7,14,21,28]:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    for w in [7,14,28]:
        df[f"roll_mean_{w}"] = df["y"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}"]  = df["y"].shift(1).rolling(w).std()
    df["dow"]         = df["ds"].dt.dayofweek
    df["month"]       = df["ds"].dt.month
    df["quarter"]     = df["ds"].dt.quarter
    df["weekofyear"]  = df["ds"].dt.isocalendar().week.astype(int)
    df["is_weekend"]  = (df["ds"].dt.dayofweek >= 5).astype(int)
    df["is_monthend"] = df["ds"].dt.is_month_end.astype(int)
    return df

In [ ]:
ts_full = pd.concat([ts_train,ts_val,ts_test]).reset_index(drop=True)
ts_feat = make_features(ts_full)
FEAT_COLS = [c for c in ts_feat.columns if c not in ["ds","y"]]
n_tr,n_va = len(ts_train),len(ts_val)
feat_train = ts_feat.iloc[:n_tr].dropna().copy()
feat_val   = ts_feat.iloc[n_tr:n_tr+n_va].dropna().copy()
feat_test  = ts_feat.iloc[n_tr+n_va:].dropna().copy()
print(f"Features: {FEAT_COLS}")

## Baseline Models

In [ ]:
def eval_fc(actual, pred, name):
    a = np.array(actual).flatten()
    p = np.array(pred).flatten()[:len(a)]
    mae  = mean_absolute_error(a, p)
    rmse = float(np.sqrt(mean_squared_error(a, p)))
    mask = a != 0
    mape = float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100) if mask.sum() else float("nan")
    print(f"{name:40s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:6.2f}%")
    return dict(model=name, MAE=mae, RMSE=rmse, MAPE=mape)

In [ ]:
results = []
naive_p = np.full(TEST_SIZE, ts_train_val["y"].iloc[-1])
results.append(eval_fc(ts_test["y"], naive_p, "Naive (last value)"))
sn = ts_train_val["y"].iloc[-7:].values
sn = np.tile(sn, TEST_SIZE//7 + 1)[:TEST_SIZE]
results.append(eval_fc(ts_test["y"], sn, "Seasonal Naive (lag-7)"))
ma = np.full(TEST_SIZE, ts_train_val["y"].iloc[-7:].mean())
results.append(eval_fc(ts_test["y"], ma, f"Moving Average (7-period)"))
print()

## LazyPredict Benchmark

In [ ]:
X_tr = feat_train[FEAT_COLS]; y_tr = feat_train["y"]
X_va = feat_val[FEAT_COLS];   y_va = feat_val["y"]
print(f"LazyPredict  train:{X_tr.shape}  val:{X_va.shape}")
try:
    lz = LazyRegressor(verbose=0, ignore_warnings=True)
    lz_models, _ = lz.fit(X_tr, X_va, y_tr, y_va)
    print("\nTop 15:")
    print(lz_models.head(15).to_string())
except Exception as e:
    print(f"LazyPredict error: {e}")

## FLAML AutoML

In [ ]:
X_tv = pd.concat([feat_train,feat_val])[FEAT_COLS]
y_tv = pd.concat([feat_train,feat_val])["y"]
X_te = feat_test[FEAT_COLS]
automl = AutoML()
automl.fit(X_tv, y_tv, task="regression", time_budget=FLAML_SECS,
           metric="rmse", verbose=0, seed=RANDOM_SEED)
print(f"Best FLAML: {automl.best_estimator}")
flaml_pred = automl.predict(X_te)
results.append(eval_fc(feat_test["y"], flaml_pred, f"FLAML ({automl.best_estimator})"))

## MLForecast — LightGBM

In [ ]:
sf_df = ts_tv_w[["ds","y"]].copy()
sf_df["unique_id"] = "delivery_total"
sf_df = sf_df[["unique_id","ds","y"]]
mlf = MLForecast(
    models=[lgb.LGBMRegressor(n_estimators=400,learning_rate=0.05,num_leaves=31,
                               random_state=RANDOM_SEED,verbose=-1)],
    freq="D", lags=[1,7,14,28],
    lag_transforms={7:[("rolling_mean",7),("rolling_std",7)],14:[("rolling_mean",14)],28:[("rolling_mean",28)]},
    date_features=["dayofweek","month","quarter"],
)
mlf.fit(sf_df)
mlf_fc = mlf.forecast(h=TEST_SIZE).reset_index(drop=True)
lgb_col = [c for c in mlf_fc.columns if c not in ["unique_id","ds"]][0]
results.append(eval_fc(ts_test["y"].values, mlf_fc[lgb_col].values, "MLForecast-LightGBM"))

context = ts_tv_w.iloc[-56:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=context["ds"],y=context["y"],name="Context",line=dict(color="#BBDEFB",width=1.5)))
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="#0D47A1",width=2.5)))
fig.add_trace(go.Scatter(x=ts_test["ds"],y=mlf_fc[lgb_col].values,
    name="MLForecast-LightGBM",line=dict(color="#F44336",width=2.5,dash="dot")))
fig.update_layout(title="MLForecast 28-day Food Delivery Forecast",template="plotly_white",
    legend=dict(orientation="h",y=-0.25))
fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("="*65)
print("ALL MODELS (ranked by RMSE)")
print("="*65)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("\nTOP 3:")
print(top3.to_string(index=False))

In [ ]:
fig = px.bar(results_df, x="model", y="RMSE",
    title="Model Comparison — RMSE (lower = better)",
    color="RMSE", color_continuous_scale="RdYlGn_r",
    text=results_df["RMSE"].round(1))
fig.update_layout(xaxis_tickangle=-35, template="plotly_white")
fig.show()

## Final Evaluation of Top 3

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="#0D47A1",width=3)))
colors3 = ["#F44336","#4CAF50","#FF9800"]
for i,(_, row) in enumerate(top3.iterrows()):
    mname = row["model"]
    if "MLForecast" in mname: pred_vals = mlf_fc[lgb_col].values
    elif "FLAML" in mname: pred_vals = flaml_pred
    else: pred_vals = sn
    fig.add_trace(go.Scatter(x=ts_test["ds"],y=pred_vals[:TEST_SIZE],
        name=f"#{i+1} {mname}  RMSE={row['RMSE']:.1f}",
        line=dict(width=2.5,dash="dot",color=colors3[i])))
fig.update_layout(title="Top 3 Models — 28-day Forecast",template="plotly_white",
    legend=dict(orientation="h",y=-0.25))
fig.show()

## Error Analysis

In [ ]:
best_name = top3.iloc[0]["model"]
if "FLAML" in best_name:
    best_pred_arr = flaml_pred
elif "sn_vals" in dir():
    best_pred_arr = sn_vals
else:
    best_pred_arr = sn

actual_v = ts_test["y"].values
nc = min(len(actual_v), len(best_pred_arr))
errors = actual_v[:nc] - best_pred_arr[:nc]

print(f"Best model   : {best_name}")
print(f"Mean error   : {errors.mean():.2f}")
print(f"Std errors   : {errors.std():.2f}")
print(f"Max abs error: {np.abs(errors).max():.2f}")

fig, axes = plt.subplots(1,3,figsize=(18,4))
axes[0].hist(errors,bins=20,edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].axvline(0,color="red",ls="--"); axes[0].set_title("Error Distribution")
axes[1].plot(range(nc),errors,"o-",ms=4,color="#F44336")
axes[1].axhline(0,color="black",ls="--",lw=1); axes[1].set_title("Errors Over Horizon")
axes[2].scatter(actual_v[:nc],best_pred_arr[:nc],s=25,alpha=0.7,color="#388E3C")
mn = min(actual_v[:nc].min(),best_pred_arr[:nc].min())
mx = max(actual_v[:nc].max(),best_pred_arr[:nc].max())
axes[2].plot([mn,mx],[mn,mx],"r--")
axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted"); axes[2].set_title("Actual vs Predicted")
plt.tight_layout(); plt.show()

## Interpretation & Insights

1. **Weekend orders peak** — Friday dinner delivery is consistently the highest single day
2. **Rainy day boost** not captured — missing weather regressors leave systematic positive residuals
3. **Pay-day effect** (1st/15th of month) creates mid-week spikes
4. **LightGBM lags capture autocorrelation** — lag_1 and lag_7 are typically top-2 features
5. **Holiday under-forecast** — delivery volume spikes on public holidays not modelled here

## Limitations

1. No weather features — rain is the strongest demand driver for food delivery
2. No price/promotion features — discounts drive ordering spikes
3. No per-restaurant breakdown — aggregate hides restaurant-level variation
4. Sample dataset may be small — time coverage could limit lag reachability
5. Platform growth trends need a trend feature to avoid systematic under-forecast

## How to Improve

1. Add daily rainfall indicator from OpenWeatherMap API
2. Include `is_payday` feature (1st and 15th of month)
3. Extend FLAML to 300 seconds
4. Add `total_cost` rolling sum as a demand intensity proxy
5. Try N-HiTS via NeuralForecast for neural zero-shot comparison

## Production Considerations

1. Retrain MLForecast daily as new orders come in
2. Build per-restaurant-type models using `unique_id` grouping
3. Set driver deployment alerts for days with forecast > 1.3× weekly mean
4. Log feature importances weekly to detect feature drift

## Common Mistakes

1. Using order time as the date (group by date, not datetime)
2. Not handling midnight orders that fall on the next calendar day
3. Not retraining after promotional campaigns (model drift)
4. Forgetting to clip Winsorisation bounds to training data only

## Mini Challenges

1. Add rainfall indicator — download daily rain data and measure RMSE drop
2. Price elasticity — does `total_cost` variance correlate with forecast error?
3. Shorter horizon — compare 7-day vs 28-day MAPE
4. Log-transform — does np.log1p reduce RMSE on this dataset?
5. Per-payday analysis — do models perform differently in week-1 vs week-3?

## Final Summary

### What We Built
- Built a daily food delivery order count series from raw Zomato records
- Validated quality, Winsorised spikes, and ran full EDA
- Benchmarked baselines, LazyPredict, FLAML, and MLForecast LightGBM
- Selected top 3 by RMSE with day-of-week error decomposition

### Key Takeaways
- Food delivery demand follows weekly cycles with Friday/weekend peaks
- LightGBM lag and calendar features reproduce seasonality effectively
- Weather and payday features are critical next steps for production accuracy
- Aggregate demand forecasting enables driver dispatch planning

---
*Zomato Orders Analysis — CC0 Public Domain*